# Conformal sensor escalation under operating-point shift — KAIST dataset

**Scope.** This notebook contains the complete analysis on the KAIST rotating-machine dataset for the study *Calibrate Where You Deploy: Conformal Sensor Escalation Under Operating-Point Shift*. It covers the cost–accuracy frontier of sensing configurations, the learning curve in windows per recording, the empirical coverage of split conformal prediction under two validation protocols, the comparison of a conformal escalation gate with a threshold gate on the top-class score at matched escalation fraction, and the recalibration experiment that quantifies how many recordings from a new operating point restore nominal coverage. The companion notebook `02_paderborn_replication.ipynb` repeats the analysis on the Paderborn KAt bearing dataset and adds the factor-wise attribution of the coverage collapse.

**Data.** KAIST testbed (Jung et al., *Data in Brief* 48:109049, 2023): 45 recordings, five condition classes (normal, inner-race fault, outer-race fault, shaft misalignment, rotor unbalance), three torque loads (0, 2 and 4 N·m), four ICP accelerometers, three current-transformer phases (only the U phase is present in every file and is the one used) and two housing temperatures; vibration and current are both sampled at 25.6 kHz (Jung et al., 2023). Severities (three bearing-fault sizes, three misalignment offsets, five unbalance masses) are pooled per class, which gives 15 recordings per load and a single healthy recording per load. After windowing: 2 250 windows. The effective sample size for any inferential statement is 45 (recordings), not 2 250.

**Reproduction.** `python run_experiments.py --dataset kaist` writes `results/*.csv` (about 10 min on a laptop); this notebook only reads those files. Figures are saved to `figs_en/`. The shared library is `cmlib.py` (cost model, classifiers, validation protocols, conformal prediction, bootstrap); `kaist_lib.py` is the dataset adapter that defines the sensing configurations.

In [ ]:
# Environment setup: runs unchanged locally (repository root as working directory) and on Google Colab.
import os, sys, subprocess
REPO_URL = "https://github.com/candidoalfilho/conformal_escalation"   # update if the repository moves
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("cmlib.py"):                       # not inside the repository: fetch it
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "conformal_escalation"], check=True)
    os.chdir("conformal_escalation")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "scikit-learn", "matplotlib", "pandas", "scipy"], check=False)
print("working directory:", os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import report_lib as R
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 3)

P  = pd.read_csv("results/pareto.csv")
LC = pd.read_csv("results/learning_curve.csv")
CC = pd.read_csv("results/conformal_coverage.csv")
G  = pd.read_csv("results/gate.csv")
RC = pd.read_csv("results/recalibration.csv")
df = pd.read_parquet("data/kaist_features.parquet")
print(f"{len(df):,} windows · {df.recording_id.nunique()} recordings · {df['class'].nunique()} classes · loads {sorted(df.load.unique())}")
display(df.groupby(["class", "load"]).recording_id.nunique().unstack().fillna(0).astype(int).rename_axis(columns="recordings per load"))

## Shared methodology

**Windowing and features.** Signals are segmented into windows of 4096 samples with a hop of 2048 (160 ms at 25.6 kHz), at most 50 windows per recording. For every channel, nine time-domain statistics are computed (mean, standard deviation, RMS, peak-to-peak, kurtosis, skewness, crest factor, shape factor, impulse factor) and, when the configuration includes the FFT, six spectral descriptors (dominant frequency, spectral centroid, spectral entropy, and band energies in 0–2 kHz, 2–6 kHz and 6–12.8 kHz). The temperature channel, which is quasi-constant within a window, contributes its mean and standard deviation. The same two functions (`tfeat`, `ffeat` in `extract_kaist.py` / `extract_paderborn.py`) are used for both datasets, so that feature vectors and costs are directly comparable.

**Sensing configurations.** A configuration is a set of channels and the choice of including the FFT. Its *cost in operations per window* is computed analytically (`kaist_lib.cost_ops`): 9 accumulations per sample for the time-domain statistics, $2.5\,N\log_2 N$ operations for a real radix-2 FFT of length $N$, plus magnitude and band accumulation per bin. The operation count isolates the arithmetic burden of each configuration on a microcontroller; it does not include acquisition energy, which is modelled separately (Section *Energy model*).

**Classifiers.** Three families are used so that any ordering of configurations is not an artefact of one learner: $\ell_2$-regularised logistic regression, a random forest (300 trees, minimum leaf size 2) and a multilayer perceptron (64–32 hidden units, tol $10^{-5}$, up to 3000 iterations). Convergence is checked for every fit and reported (`converged = 1.00` throughout).

**Validation protocols.** Windows from the same recording are strongly dependent (same mounting, thermal state, residual unbalance). All splits are therefore grouped by recording. Two protocols are used: *recording-grouped stratified 3-fold cross-validation*, repeated over three seeds, which estimates in-distribution performance; and *leave-one-operating-point-out* (LOLO), which trains on all but one operating point and tests on the held-out one, which estimates performance under operating-point shift. Confidence intervals are 95 % percentile intervals from a cluster bootstrap that resamples *recordings* (600 replicates); a window-level bootstrap would understate uncertainty by roughly an order of magnitude.

**Split conformal prediction.** From the training recordings of each fold, a stratified third is carved out as a calibration set (never used for fitting). With non-conformity score $s_i = 1 - \hat p(y_i \mid x_i)$ (LAC) or the cumulative sorted probability up to the true class (APS), the conformal quantile $\hat q$ is the $\lceil (n+1)(1-\alpha)\rceil$-th smallest calibration score (`cmlib.conformal_qhat`; $\hat q = 1$ when that rank exceeds $n$) and the prediction set is $C(x) = \{y : s(x, y) \le \hat q\}$. Under exchangeability of calibration and test data, $\mathbb P\,(Y \in C(X)) \ge 1 - \alpha$ for any classifier. Two qualifications apply. First, windows of one recording are dependent, so exchangeability holds at recording level and the finite-sample statement is approximate, with an effective calibration size equal to the number of calibration recordings (hierarchical conformal prediction; Dunn, Wasserman & Ramdas, *JASA* 2023; Lee, Barber & Willett, 2023). Second, in the coverage and gate experiments (E3, E4, X1) the calibration scores of the three folds are pooled and a single $\hat q$ is applied to the pooled test predictions; the recalibration experiments (E5, E6, X3) use one quantile per held-out operating point. Coverage is reported as the empirical fraction of test windows whose true label lies in $C(x)$; the *gap* is coverage minus the nominal $1-\alpha$. In weighted conformal prediction (E6) calibration score $i$ carries a weight $w_i$, the test point weight 1, and $\hat q$ is the $(1-\alpha)$-quantile of the weighted empirical distribution normalised by $\sum_i w_i + 1$ (`cmlib.weighted_conformal_qhat`); with $w_i \equiv 1$ it reduces to the unweighted quantile, and for fixed weights the applicable guarantee is the total-variation bound of Barber et al. (2023).

**The escalation gate.** The cheap configuration is always on. A window is *escalated* to the expensive configuration if and only if its conformal set is not a singleton. The risk level $\alpha$ is the only parameter; the escalation fraction is measured, not tuned. The baseline *threshold gate* escalates when the top-class score $\max_y \hat p(y \mid x)$ falls below a swept threshold $\tau$. With the LAC score, $C(x)$ is a singleton iff the top-class probability is at least $1-\hat q$ and the second-highest is below it; when $1-\hat q > 0.5$ this coincides with a threshold on the top-class score, but at the quantiles observed here ($\hat q$ mostly above 0.5) the two gates are different selection rules, and the comparison is empirical. They are compared *at matched escalation fraction*: for every conformal operating point the threshold-gate curve is linearly interpolated at the same escalation fraction and the difference in macro-F1 is reported.

**Recalibration under shift.** For each held-out operating point, the model is trained on the remaining points and three calibration sets are compared: the *source* calibration recordings only; the source set *pooled* with $k$ labelled recordings from the target point; and the $k$ target recordings *only*. Test recordings are always disjoint from calibration recordings. Results are averaged over five seeds (random choice of the $k$ recordings).

### Sensing configurations and their cost

Nine configurations are evaluated. Two of them have identical operation counts but require different hardware (one accelerometer with FFT versus one current phase with FFT), which is why the energy model below is needed in addition to the operation count.

In [ ]:
import kaist_lib as K
blocks = K.feature_blocks(df)
tab = pd.DataFrame([(R.KAIST_BLOCKS[b][0], R.KAIST_SHORT[b], len(cols), int(K.cost_ops(spec))) for b, (cols, spec) in blocks.items()],
                   columns=["configuration", "abbrev.", "n_features", "ops_per_window"]).set_index("configuration")
display(tab)

## Energy model

Operation counts are the wrong currency for a battery-powered node: acquisition dominates. The table lists datasheet power figures for the components of a deployable node (ESP32-S3 with a digital MEMS accelerometer and an isolated current transducer) and, for comparison, the laboratory chain used by the benchmark testbeds. Energy per 160 ms window is power × window duration for the sensing chain. The current transducer figure includes the secondary-side term $I_C + I_P/N_S$ at 5 V for the Paderborn motor (2.3 A, $N_S = 1731$); ignoring that term gives 75 mW typical / 105 mW maximum.

Two consequences follow. First, **one current phase costs ≈25× the energy of a three-axis MEMS accelerometer per window, and two phases ≈50×**: the cascade "cheap current, expensive vibration" is inverted when the currency is joules. Second, the arithmetic is negligible by comparison: a 4096-point FFT on the ESP32-S3 costs about 19 µJ (34 400 cycles extrapolated from the ESP-DSP benchmark table, which stops at 1024 complex points, × 543 pJ per cycle in single-core SIMD mode) against 0.5 mJ to keep the MEMS accelerometer powered during the window. The extrapolated cycle count is the one quantity in this model that is derived rather than measured and is marked as such.

Three caveats apply. The figures model a *dedicated, active* transducer; a passive split-core transformer or Rogowski coil draws no supply power and would make current the cheap tier again. A drive inverter that already samples phase current for control delivers that signal at near-zero marginal energy; in that scenario current is free, but the signal is sampled at the control-loop rate with lower resolution than the laboratory signal, so the current-tier models would have to be trained on inverter-sampled data. Finally, the MCU is not idle during acquisition: the IIS3DWB FIFO holds about an eighth of a 4096-sample three-axis window, so the ESP32-S3 sleeps between eight FIFO drains over SPI (about 20 ms of bus time at the idle floor), ≈2.2 mJ per window or four times the sensor itself (`energy_params.mcu_acquisition_mj`); the current ADC has no FIFO, so on a dedicated current tier the MCU idles in WAITI for the whole window (≈17 mJ). The computation figure is also given on a total-power basis (≈0.07 mJ) that includes the idle floor a duty-cycled node pays. At one window per minute the MEMS tier with its MCU service averages about 46 µW against a 26 µW deep-sleep floor, so the acquisition energy of the cascade matters for near-continuous monitoring.

In [ ]:
import energy_params as E
et = R.energy_table(E); display(et)
w = 4096/25600
mems, lem = E.IIS3DWB["p_mw"], E.LEM_CKSR_15NP["p_mw"]
print(f"current/MEMS energy ratio per window: {lem/mems:.1f}x per phase, {2*lem/mems:.1f}x for two phases")
print(f"FFT 4096 on ESP32-S3 (extrapolated cycles): {E.fft_energy_uj():.1f} uJ  vs  MEMS sensing per window: {mems*w*1e3:.0f} uJ")

## 1. Cost–accuracy frontier (E1)

Macro-F1 of every configuration and classifier under the two protocols. Colour encodes the sensing family, marker shape the classifier; error bars are 95 % cluster-bootstrap intervals over recordings. The grey line joins the Pareto-efficient configurations (best classifier per configuration). The temperature-only configuration (4 operations per window) is omitted from the plot and reported in the table: it is a confounder rather than a candidate configuration, as the shift protocol shows.

In [ ]:
R.fig_pareto(P, R.KAIST_BLOCKS, R.KAIST_SHORT, name="kaist_fig1_pareto");
display(R.tab_pareto(P, R.KAIST_BLOCKS))

**Observations.** (i) A single accelerometer with nine time-domain features (36 864 operations per window) reaches a macro-F1 of 0.69 under recording-grouped validation and 0.69 under load shift, against 0.73 / 0.75 for four accelerometers with FFT (671 744 operations): an 18-fold reduction in arithmetic and the elimination of three ICP conditioning chains for a loss of about 0.04. (ii) The FFT is dispensable in this dataset: four accelerometers without FFT lose 0.002–0.010 relative to the full configuration, within the confidence interval. (iii) Motor current alone does not sustain the diagnosis: 0.42 under grouped validation and 0.21 under load shift; it is the only configuration whose performance *collapses* rather than degrades. (iv) Temperature behaves as a recording identifier rather than a feature: 0.52 under grouped validation falls to 0.18 under load shift. Grouped validation does not protect against this confounder; the shift protocol exposes it. (v) Adding current to the four accelerometers does not improve on vibration alone.

## 2. Learning curve in windows per recording (E2)

Feature extraction stopped at 50 windows per recording. If performance had not saturated, the natural objection "too few windows" would call for re-extraction. The curve below uses 10 to 50 windows per recording with the same grouped protocol.

In [ ]:
R.fig_learning_curve(LC, R.KAIST_BLOCKS, name="kaist_fig2_learning_curve");
display(LC.pivot_table(index="n_windows", columns=["block", "model"], values="f1_mean").round(3))

The curve is flat (variation below 0.03 across all configurations, within seed-to-seed noise). Additional windows from the same recordings add no information; the binding constraint is the number of recordings, which motivates the second dataset.

## 3. Empirical coverage of split conformal prediction (E3)

The cheap tier of the cascade (current, time+FFT) is calibrated on recordings held out from training and evaluated on unseen recordings, under both protocols, for both scores and eight values of α. Panel (a) plots empirical against nominal coverage; panel (b) breaks the shift protocol down by held-out torque load at nominal 80 %.

In [ ]:
R.fig_coverage(CC, per_group_label=lambda c: c.replace("cov_", "").replace("Nm", " N·m"), name="kaist_fig3_coverage");
display(R.tab_coverage_gap(CC))
print("coverage per class at nominal 80 % (LAC), mean over classifiers and seeds; class 0 = normal")
sel = CC[(CC.score=="lac") & np.isclose(CC.alpha, 0.20)]
display(sel.groupby("protocol")[[c for c in CC.columns if c.startswith("covclass_")]].mean().round(3))

**Under exchangeability the guarantee holds.** With recording-grouped validation the mean gap is +0.045 (LAC) and +0.085 (APS): conservative, as the finite-sample correction and the grouping predict.

**Marginal coverage is not class-conditional coverage.** Even in distribution, the normal class (one recording per load, three in total) is covered far below nominal while the fault classes are covered above it; the marginal guarantee averages over classes, so a false-alarm bound on the healthy class needs class-conditional (Mondrian) calibration, which the healthy-class recordings alone would support.

**Under load shift it fails.** With leave-one-load-out validation the mean LAC gap is −0.148 and the worst case −0.453. At nominal 80 % the gate delivers 55.3 % coverage, and the failure is uneven across operating points (0.41 at 0 N·m, 0.73 at 2 N·m, 0.52 at 4 N·m). APS is less affected on average (−0.013) at the price of much larger sets. This is not a defect of the implementation: the exchangeability assumption between calibration and test recordings is violated by construction, and the measurement quantifies how much of the guarantee is lost.

## 4. Conformal gate versus threshold gate (E4)

Two cascades are evaluated: *current (time+FFT) → four accelerometers (time+FFT)* and *one accelerometer (time) → four accelerometers (time+FFT)*. For each, the threshold τ on the top-class score is swept and the conformal gate is run for the eight values of α with both scores. The comparison is made at matched escalation fraction; a positive difference favours the conformal gate.

In [ ]:
CASC = {"cur_full->vib_full": "current (time+FFT) → 4 accelerometers (time+FFT)",
        "vib1_time->vib_full": "1 accelerometer (time) → 4 accelerometers (time+FFT)"}
R.fig_gate(G, CASC, model="rf", name="kaist_fig4_gate");
display(R.tab_gate_delta(G, protocol="grouped"))
display(G[G.protocol=="grouped"].groupby("cascade")[["cheap_ops","exp_ops","f1_cheap","f1_exp"]].mean().round(3))

**The conformal gate does not buy accuracy.** The mean difference in macro-F1 at matched escalation fraction is −0.000 ± 0.004 over all cascades, scores and seeds; the two gates lie on the same accuracy–escalation curve although, at the quantiles observed, they are different selection rules (see *Shared methodology*). What the conformal gate changes is the *specification*: τ is a dimensionless knob tuned on a validation set, whereas α is a stated risk level from which the escalation fraction, and therefore the energy budget, follows as a measured consequence.

**Which cascade.** The current-first cascade has a 4× ratio in operations and a large accuracy gap between tiers (0.35 → 0.68), but the cheap tier is so weak that the gate escalates 87–100 % of windows at any α ≤ 0.20 and saves nothing. The accelerometer-first cascade has an 18× ratio and a small gap (0.63 → 0.68); at α = 0.20 it escalates 12.3 % of windows for a macro-F1 of 0.655 at a mean cost of 120 k operations against 672 k for the always-on expensive tier. In energy terms, however, the accelerometer-first cascade keeps one accelerometer powered at all times; the two cascades optimise different currencies, which is why the operation count alone cannot decide between them.

## 5. Recalibration at the target operating point (E5)

The engineering question that follows from Section 3 is how much labelled data from the new operating point is needed to recover the guarantee. For each held-out load, $k \in \{0, 1, 2, 3, 5, 8\}$ target recordings enter the calibration set either pooled with the source calibration set or replacing it; test recordings are disjoint from calibration recordings (variant `rec`). A second variant (`win`) calibrates on the first $m$ windows of *each* target recording and tests on the remaining windows of the same recordings; it corresponds to "run the machine briefly and label", but calibration and test share recordings, so its coverage is optimistic. The contrast between the two variants is itself informative.

In [ ]:
R.fig_recal(RC, group_label=lambda s: f"target {s.replace('Nm',' N·m')}", name="kaist_fig5_recalibration");
print("variant 'rec' (disjoint recordings) — gap by k and calibration mix, mean over both classifiers and α ∈ {0.05, 0.10, 0.20, 0.30}")
display(R.tab_recal(RC))
print("same, per classifier at α = 0.20")
display(pd.concat({m: R.tab_recal(RC, model=m, alpha=0.20) for m in ["logreg", "rf"]}, axis=1))
print("variant 'win' (first m windows of each target recording in calibration; test on the remaining windows of the same recordings)")
d = RC[RC.variant=="win"].assign(gap=lambda d: d.coverage-d.nominal)
display(d.groupby(["k", "model"])["gap"].mean().unstack().assign(both=lambda x: x.mean(axis=1)).round(3).rename_axis(index="m (windows per recording)"))

**Five recordings from the new operating point restore nominal coverage.** Averaged over both classifiers and the four risk levels, the gap moves from −0.144 with source calibration only to −0.001 with $k=5$ and +0.010 with $k=8$. This is a commissioning quantity: five short runs at the new operating point, labelled, and the guarantee holds again. The per-classifier view shows that the random forest is far less affected by the shift at α = 0.20 (source gap −0.05) than the logistic regression (−0.41), so the number of recordings needed depends on how over-confident the classifier is off-distribution; the aggregate is the conservative figure to plan with.

**Discarding the source calibration set is better than pooling it.** At $k=3$, calibrating on the three target recordings alone gives a gap of −0.052, while pooling them with the source set gives −0.100, twice as bad. Stale non-conformity scores contaminate the quantile; what matters for the guarantee is exchangeability, not volume. The principled alternative, down-weighting rather than discarding the source scores (Barber et al., 2023), is evaluated in Section 6.

**Calibrating on the first windows of each run recovers less than calibrating on whole runs.** Variant `win` with $m = 5, 10, 20$ windows per recording reaches gaps of −0.078, −0.071 and −0.056, against −0.001 for five whole recordings, even though calibration and test then share recordings. The early windows of a run are evidently not exchangeable with its later windows (start-up transient, thermal drift), which argues for calibration recordings that span the run rather than its first seconds.

## 6. Weighted conformal prediction as the alternative to discarding the source set (E6)

Section 5 replaced the source calibration set by the target recordings. The principled alternative is to keep the source scores with a reduced weight (weighted conformal prediction; Tibshirani et al., 2019; Barber et al., 2023). Here the source scores receive a fixed weight $\rho$ and the target scores weight 1, so that $\rho = 1$ is ordinary pooling and $\rho = 0$ is target-only calibration; intermediate values are the fixed-weight schemes a practitioner could set without estimating likelihood ratios. Same five seeds, recordings and α grid as Section 5, so that ρ = 1 reproduces the pooled column and ρ = 0 the target-only column of that section (to within 0.01).

In [ ]:
W = pd.read_csv("results/weighted_recalibration.csv")
R.fig_weighted(W, name="kaist_fig6_weighted");
g, s = R.tab_weighted(W); print("coverage gap"); display(g); print("mean set size"); display(s)

Down-weighting interpolates between the two extremes and confirms the ordering of Section 5: pooling ($\rho = 1$) is the worst option at every $k \ge 2$, and any $\rho \le 0.3$ recovers most of the coverage of target-only calibration. At $k = 1$ a small residual weight on the source scores ($\rho \approx 0.1$–$0.3$) is better than both extremes, because a single target recording gives a very noisy quantile and the source scores stabilise it. The practical rule is therefore not "discard" but "down-weight heavily": $\rho \le 0.1$ once two or more target recordings are available.

## 7. Summary of findings on KAIST

1. One accelerometer with time-domain features is within 0.04 macro-F1 of four accelerometers with FFT at 1/18 of the arithmetic; the FFT is dispensable; current alone collapses under load shift; temperature is a recording-level confounder revealed only by the shift protocol.
2. Performance is saturated in windows per recording; the limiting quantity is the number of recordings.
3. Split conformal coverage holds under recording-grouped validation (+0.045) and collapses under load shift (−0.148 mean, −0.453 worst; 80 % nominal → 55 %); marginal coverage hides a healthy-class coverage of 38 % even in distribution.
4. The conformal gate and a tuned threshold gate are indistinguishable in accuracy at matched escalation fraction (Δ = −0.000 ± 0.004); the conformal gate's contribution is a stated risk level in place of a tuned threshold.
5. Five target-point recordings restore coverage; pooling them with the stale source calibration set is worse than replacing it, and weighted conformal prediction with a small source weight (ρ ≤ 0.1) matches target-only calibration while stabilising the quantile when only one target recording is available.

**Limitations specific to this dataset.** 45 recordings and three loads leave about ten recordings in each calibration set under the shift protocol, so coverage estimates are noisy and the recalibration curve rests on small $k$. The three loads vary torque only, so the collapse cannot be attributed to a specific physical factor. Both limitations are addressed with the Paderborn dataset in the companion notebook.